# API Architecture Styles & Communication Protocols

An **API (Application Programming Interface)** is a set of rules, definitions, and protocols that enables one software application to communicate and exchange data with another. Choosing the right API architecture and communication protocol is a fundamental decision in distributed systems and system design interviews.

![API Architecture Styles](api-architecture-style.png)

---

## Quick Reference: Architecture & Protocol Comparison

| Protocol / Style | Transport | Payload Format | Communication Pattern | Best Use Case |
| :--- | :--- | :--- | :--- | :--- |
| **REST** | HTTP/1.1, HTTP/2 | JSON, XML | Request-Response (Stateless) | Public CRUD APIs, Web & Mobile backends |
| **GraphQL** | HTTP/1.1, HTTP/2 | JSON | Request-Response, Subscriptions | Complex UI frontends, flexible data requirements |
| **gRPC** | HTTP/2 | Protocol Buffers (Binary) | Unary, Server/Client/Bidi Streaming | High-throughput internal Microservices, Low-latency IPC |
| **SOAP** | HTTP, SMTP, TCP | XML | Request-Response (Strict Contract) | Legacy Enterprise systems, Banking, WS-Security |
| **WebSocket** | TCP (Upgraded HTTP) | Text, Binary | Full-Duplex Bidirectional | Real-time chat, multiplayer gaming, financial trading |
| **SSE** | HTTP/1.1, HTTP/2 | Plain text (`text/event-stream`) | Unidirectional (Server-to-Client) | Live notifications, LLM token streaming (ChatGPT), Stock feeds |
| **Webhook** | HTTP POST | JSON, XML | Event-driven Push (Asynchronous) | Third-party event triggers (Stripe, GitHub, PayPal) |
| **Long-Polling** | HTTP/1.1 | JSON, XML | Pseudo-streaming via held requests | Fallback real-time updates when WebSockets unavailable |
| **Short-Polling** | HTTP/1.1 | JSON, XML | Periodic Request-Response | Low-priority status checks with relaxed freshness |
| **MQTT** | TCP | Binary (Compact) | Publish-Subscribe (Broker-based) | IoT devices, smart home, low-bandwidth/unreliable networks |

---

## 1. REST (Representational State Transfer)

REST is an architectural style designed around **resources** identified by URIs. It leverages standard HTTP semantics and is the dominant API style for web and mobile backends.

### Core Principles:
- **Statelessness**: Every request contains all necessary authentication and state data; the server stores no client session context.
- **Standard HTTP Methods**:
  - `GET`: Safe and Idempotent retrieval of resources.
  - `POST`: Non-idempotent creation of a subordinate resource.
  - `PUT`: Idempotent full replacement of a resource.
  - `PATCH`: Partial modification of a resource.
  - `DELETE`: Idempotent removal of a resource.
- **Cacheability**: Built-in HTTP caching using `Cache-Control`, `ETag`, and `Last-Modified` headers.
- **Uniform Interface**: Consistent resource identifiers (e.g., `/api/v1/orders/123`).

### Advantages:
- Simple to understand, build, and debug using standard browsers and tools (cURL, Postman).
- Highly scalable due to stateless caching layers (CDNs, Reverse Proxies).

### Disadvantages & Limitations:
- **Over-fetching**: Returns all fields of a resource even when the client only needs a few.
- **Under-fetching / N+1 Problem**: Clients often need multiple sequential round trips to fetch related resources (e.g., `/users/1` then `/users/1/orders` then `/orders/99/items`).

---

## 2. GraphQL

Developed by Facebook in 2012, GraphQL is a query language and server-side runtime for APIs that provides clients the power to ask for **precisely what they need and nothing more**.

### Core Concepts:
- **Single Endpoint**: Typically exposed via a single HTTP `POST /graphql` endpoint.
- **Schema Definition Language (SDL)**: Strongly typed schema defining Object Types, Fields, and Relationships.
- **Operations**:
  - `Query`: Read data (equivalent to GET).
  - `Mutation`: Modify data (equivalent to POST, PUT, DELETE).
  - `Subscription`: Real-time event push over WebSocket.

### Advantages:
- Solves both **Over-fetching** and **Under-fetching** in a single round-trip.
- Strongly typed contract facilitates automatic code generation for web and mobile clients.
- Evolve APIs without versioning (deprecate specific fields rather than releasing `/v2`).

### Disadvantages:
- **Complex Caching**: HTTP caching based on URL paths (`GET /users/1`) cannot be used easily since all requests hit `/graphql` via POST.
- **N+1 Database Query Problem**: Naive resolvers can execute $1 + N$ SQL queries. Requires batching and caching mechanisms (e.g., **DataLoader** pattern).
- **Performance Vulnerabilities**: Malicious or recursive queries can overwhelm database resources unless query complexity and depth limiters are enforced.

---

## 3. gRPC (Google Remote Procedure Call)

gRPC is a high-performance, open-source universal RPC framework developed by Google. It enables applications to call methods on remote servers as if they were local in-memory function calls.

### Core Characteristics:
- **Protocol Buffers (Protobuf)**: Uses Protobuf binary serialization instead of text-based JSON/XML, resulting in 5x-10x smaller payload size and 7x-10x faster serialization/deserialization.
- **Built on HTTP/2**:
  - Multiplexing (`Sends multiple requests and responses concurrently over a single TCP connection, eliminating head-of-line blocking.`) multiple requests over a single TCP connection.
  - Binary framing and HPACK header compression.
- **Four Streaming Modes**:
  1. **Unary RPC**: Single request $\rightarrow$ single response. The client sends a single request and gets back a single response, standard request-response model.
  2. **Server Streaming RPC**: Single request $\rightarrow$ stream of responses. The client sends a single request and receives a continuous stream of responses until the server closes the stream.
  3. **Client Streaming RPC**: Stream of requests $\rightarrow$ single response. The client sends a stream of requests and receives a single summary response once the server processes the full stream.
  4. **Bidirectional Streaming RPC**: Simultaneous, independent streams in both directions. Both client and server send independent, concurrent streams of messages over a single connection.

### Best For:
- Internal service-to-service communication in Microservice architectures.
- Low-latency, high-throughput systems, and polyglot microservice environments.

### Limitations:
- Limited native web browser support (requires `gRPC-Web` proxy).
- Binary payloads are not human-readable without schema tools.

---

## 4. SOAP (Simple Object Access Protocol)

SOAP is a standardized, XML-based messaging protocol specification for exchanging structured information. It relies on strict contracts and application protocols such as HTTP, SMTP, or TCP.

### Key Features:
- **WSDL (Web Services Description Language)**: Explicit XML-based contract strictly describing all available operations, parameters, and data types.
- **WS-Security**: Enterprise-grade security standards with message-level encryption, digital signatures, and token-based authentication.
- **ACID Transaction Support**: Supports distributed ACID transactions via WS-AtomicTransaction.

### When to Use:
- Enterprise banking, financial transaction gateways, healthcare systems, and legacy integrations requiring strict compliance and guaranteed transactional delivery.

### Drawbacks:
- Verbose XML syntax results in heavy bandwidth consumption.
- High parsing overhead and rigid contracts compared to modern REST/JSON.

---

## 5. Webhook (Reverse API / Event-Driven Push)

A **Webhook** is an automated HTTP `POST` notification sent from a source system to a destination system immediately when a specific business event occurs. Unlike polling, webhooks push data in real time.

### Architecture & Security Flow:

![Webhook Security & Lifecycle Flow](images/webhook-security-flow.svg)

### Advantages:
1. **Near Instantaneous**: Zero polling delay; data arrives when created.
2. **Minimal Server Overhead**: No wasted bandwidth on empty poll cycles.

### Failure Handling & Reliability Invariants:
1. **Signature Verification (HMAC-SHA256)**: Prevent spoofing by verifying secret signatures in request headers (e.g., `X-Signature`).
2. **Idempotency**: Webhook providers operate on *at-least-once delivery*. Handlers must verify unique event IDs (`eventId`) before processing.
3. **Asynchronous Ingestion**: Never perform slow database queries or email sending directly inside the webhook HTTP handler. Immediately push to a queue (Kafka, RabbitMQ, SQS) and return `200 OK`.
4. **Reconciliation**: Periodically execute fallback batch reconciliation jobs to detect missed webhooks.



## 6. WebSocket (Full-Duplex Bidirectional Communication)

WebSocket provides a persistent, full-duplex, bidirectional communication channel over a single TCP connection, standardized in RFC 6455.

### Lifecycle:
1. **HTTP Handshake**: Client sends an HTTP GET request with upgrade headers:
   ```http
   Connection: Upgrade
   Upgrade: websocket
   Sec-WebSocket-Key: dGhlIHNhbXBsZSBub25jZQ==
   ```
2. **Server Response**: Server acknowledges with `HTTP 101 Switching Protocols`.
3. **Bidirectional Framing**: Both sides can send binary or text frames asynchronously at any time with minimal 2-to-10 byte framing overhead (no HTTP headers per message).
4. **Heartbeat (Ping/Pong)**: Periodic keep-alive pings prevent idle connection drops by intermediate proxies.

### Best For:
- Collaborative editing (Google Docs, Figma), multiplayer gaming, live chat applications, financial order book tickers.

---

## 7. Server-Sent Events (SSE)

Server-Sent Events (SSE) is a standardized web technology (HTML5) that enables a client to establish a persistent, long-lived HTTP connection and receive asynchronous data updates pushed from the server.

### Key Differences vs WebSocket:
- **Unidirectional**: Server pushes data to client; client does not send messages through the SSE stream.
- **Transport**: Standard HTTP (`Content-Type: text/event-stream`). Works seamlessly through firewalls, reverse proxies, and HTTP/2 multiplexing.
- **Native Browser Support**: Built-in `EventSource` API in all modern browsers.
- **Automatic Reconnection**: Browsers automatically reconnect with the `Last-Event-ID` header if the connection drops.

### Best For:
- **LLM / AI Token Streaming** (ChatGPT streaming responses).
- Real-time dashboards, live social media notification feeds, stock ticker updates.

---

## 8. Polling: Short Polling vs. Long Polling

| Feature | Short Polling | Long Polling |
| :--- | :--- | :--- |
| **Mechanism** | Client requests at fixed interval (e.g., every 2s). | Client sends request; server holds open until data arrives or timeout. |
| **Latency** | High (delay up to polling interval). | Low (server responds immediately upon event). |
| **Server Overhead** | Very high (95%+ requests return empty `304` or `{}`). | Moderate (many open connection file descriptors). |
| **Connection Model** | Immediate disconnect after each response. | Immediate reconnect after each response. |
| **Best For** | Low-frequency checks where real-time is not critical. | Fallback when WebSockets/SSE are blocked by corporate proxies. |

---

## 9. MQTT (Message Queuing Telemetry Transport)

MQTT is an extremely lightweight, publish-subscribe network protocol designed for constrained devices, low-bandwidth, high-latency, or unreliable networks (IoT).

### Core Architecture:
- **Publish/Subscribe via Broker**: Clients publish messages to hierarchical topics (e.g., `sensors/temp/room1`); subscribers receive messages matching topic filters.
- **Minimal Packet Overhead**: Fixed header is only **2 bytes**, making it ideal for battery-operated devices.
- **Quality of Service (QoS) Levels**:
  - **QoS 0 (At most once)**: Fire-and-forget; no delivery guarantee.
  - **QoS 1 (At least once)**: Guaranteed arrival with acknowledgment, but duplicates possible.
  - **QoS 2 (Exactly once)**: 4-step handshake ensuring message is delivered exactly once without duplicates.
- **Last Will and Testament (LWT)**: Broker notifies subscribers if a device disconnects unexpectedly.

---

## 10. Message Brokers & Asynchronous Event Streaming

In large distributed architectures, point-to-point synchronous APIs are supplemented or replaced by message queues and event streaming platforms.

### Point-to-Point Queue (e.g., RabbitMQ, AWS SQS):
- **Producer** pushes message to queue; exactly **one consumer** processes each message.
- Decouples heavy workloads (e.g., video transcoding, PDF generation, email dispatch).

### Event Streaming / Log-Based (e.g., Apache Kafka, Redpanda):
- Append-only, partitioned distributed commit log.
- **Publish-Subscribe**: Multiple independent consumer groups can read the same stream at their own pace.
- High throughput (millions of events/sec), event replayability, and stateful stream processing.

---

## 11. HTTP Protocol Evolution: HTTP/1.1 vs HTTP/2 vs HTTP/3

| Feature | HTTP/1.1 (1997) | HTTP/2 (2015) | HTTP/3 (2022) |
| :--- | :--- | :--- | :--- |
| **Transport** | TCP | TCP | **QUIC (over UDP)** |
| **Multiplexing** | No (Head-of-Line blocking) | Yes (Binary streams over 1 TCP) | Yes (Independent streams over UDP) |
| **Packet Loss Impact** | Affects individual connection | TCP HoL blocking halts all streams | **Zero HoL blocking** (other streams continue) |
| **Format** | Plain Text | Binary Framing | Binary Framing |
| **Header Compression** | None | HPACK | QPACK |
| **Handshake Latency** | TCP + TLS (2-3 RTTs) | TCP + TLS (2-3 RTTs) | **0-RTT or 1-RTT Connection Setup** |
| **Connection Migration** | Breaks on IP change (WiFi $\rightarrow$ 4G) | Breaks on IP change | **Smooth migration** via Connection ID |

---

## 12. Real-Time Communication Protocol Decision Framework

When deciding on a communication protocol in a System Design interview, apply this decision framework:

```
Do you need real-time communication?
 ├── NO  ──► Standard REST or GraphQL
 └── YES
      ├── Is communication bidirectional (both client & server send data frequently)?
      │    ├── YES ──► WebSocket (Chat, Gaming, Whiteboards)
      │    └── NO  (Only server pushes data to client)
      │         ├── Need simple browser-friendly event stream? ──► Server-Sent Events (SSE) (ChatGPT, Stock alerts)
      │         └── Inter-system / 3rd-party asynchronous trigger? ──► Webhook
      ├── Is it IoT or resource-constrained devices? ──► MQTT
      └── Is it high-throughput microservice-to-microservice? ──► gRPC
```
